# Advanced Programming for Data Science
## TP2

## 0. Setup and Initial Data Exploration

In [16]:
import os
import time
import functools
from abc import ABC, abstractmethod
from typing import Optional, List
import pandas as pd
import numpy as np
from IPython.display import display

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:   {np.__version__}")

Pandas version: 2.3.2
NumPy version:   1.26.4


### Inspecting `sample_data.csv`
Let's inspect the target dataset `sample_data.csv` to see its structure, data types, and missing values.

In [17]:
# Inspect sample_data.csv directly
raw_df = pd.read_csv("sample_data.csv")
print(f"Dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print("\nMissing value count per column:")
print(raw_df.isna().sum())
print("\nDataset contents:")
display(raw_df)

Dataset shape: 7 rows, 7 columns

Missing value count per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Dataset contents:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


---
## Exercise 1: Build the `CSVReader` Class (OOP Foundation)

### Objectives & Concepts
- **Encapsulation:** Keep file state (`file_path`, loaded `data`) within the class and expose clear methods (`read`, `preview`).
- **Abstraction:** The caller interacts with clean interfaces without dealing with low-level file stream management.

### Instructions:
1. Create a class `CSVReader`.
2. Define an `__init__` method taking `file_path`.
3. Add a `read()` method that loads the CSV file using pandas and stores it in `self.data`.
4. Add a `preview(n)` method that prints / displays the top $n$ rows.

In [18]:
class CSVReader:
    def __init__(self, file_path: str):
        if not isinstance(file_path, str) or not file_path.strip():
            raise ValueError("file_path must be a non-empty string.")
        self.file_path = file_path
        self.data: Optional[pd.DataFrame] = None

    def read(self) -> pd.DataFrame:
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n: int = 5) -> pd.DataFrame:
        if self.data is None:
            print("Data not yet read. Reading file now...")
            self.read()
        print(f"=== Preview (First {n} rows of '{self.file_path}') ===")
        subset = self.data.head(n)
        display(subset)
        return subset

In [19]:
# Testing CSVReader on sample_data.csv
reader = CSVReader("sample_data.csv")
df_loaded = reader.read()
print(f"Loaded DataFrame with shape: {df_loaded.shape}")
reader.preview(5)

Loaded DataFrame with shape: (7, 7)
=== Preview (First 5 rows of 'sample_data.csv') ===


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


---
## Exercise 2: Apply Strategy Pattern for Missing Values Handling

### Objectives & Concepts
- **Strategy Pattern:** Defines a family of algorithms, encapsulates each one, and makes them interchangeable. Strategy lets the algorithm vary independently from clients that use it.
- **Open/Closed Principle (OCP):** Classes should be open for extension, but closed for modification. We can add new imputation methods (e.g. `FillMedian`, `KNNImpute`) without modifying `DataCleaner`.
- **Polymorphism:** `DataCleaner` delegates to any class implementing `MissingValueStrategy.handle(df)`.

### Architecture:
```
       +-----------------------+
       | MissingValueStrategy  |  <<Interface / ABC>>
       +-----------------------+
       | + handle(df)          |
       +-----------------------+
          ^        ^        ^
          |        |        |
+-------------+ +----------+ +----------+
| DropMissing | | FillMean | | FillMode |
+-------------+ +----------+ +----------+

       +------------------------+
       |      DataCleaner       |
       +------------------------+
       | - strategy             |
       | + set_strategy(...)    |
       | + clean(df)            |
       +------------------------+
```

### Instructions:
1. Create an abstract class `MissingValueStrategy` with abstract method `handle(df)`.
2. Create concrete subclasses:
   - `DropMissing`: drops any rows with missing values (`df.dropna()`).
   - `FillMean`: fills missing values in numeric columns with the column mean.
   - `FillMode`: fills missing values with each column's mode.
3. Create a `DataCleaner` class that applies the chosen strategy.

In [20]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.copy()
        numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if cleaned_df[col].isna().any():
                col_mean = cleaned_df[col].mean()
                cleaned_df[col] = cleaned_df[col].fillna(col_mean)
        return cleaned_df


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.copy()
        for col in cleaned_df.columns:
            if cleaned_df[col].isna().any():
                col_mode = cleaned_df[col].mode()
                if not col_mode.empty:
                    cleaned_df[col] = cleaned_df[col].fillna(col_mode.iloc[0])
        return cleaned_df


class DataCleaner:
    def __init__(self, strategy: Optional[MissingValueStrategy] = None):
        self._strategy = strategy

    @property
    def strategy(self) -> Optional[MissingValueStrategy]:
        return self._strategy

    @strategy.setter
    def strategy(self, strategy: MissingValueStrategy):
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("Strategy must be an instance of MissingValueStrategy.")
        self._strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        if self._strategy is None:
            raise ValueError("No strategy configured! Call set_strategy() before clean().")
        return self._strategy.handle(df)

In [21]:
# Testing Strategy Pattern on sample_data.csv
cleaner = DataCleaner()

print("Original Data Missing Values:")
print(df_loaded.isna().sum().to_dict())

# 1. Strategy: DropMissing
cleaner.set_strategy(DropMissing())
df_dropped = cleaner.clean(df_loaded)
print(f"\n--- Strategy 1: DropMissing --- (Shape: {df_dropped.shape})")
display(df_dropped)

# 2. Strategy: FillMean
cleaner.set_strategy(FillMean())
df_mean = cleaner.clean(df_loaded)
print(f"\n--- Strategy 2: FillMean --- (Shape: {df_mean.shape})")
print("Remaining numeric NaNs:", df_mean.select_dtypes(include=[np.number]).isna().sum().to_dict())
display(df_mean)

# 3. Strategy: FillMode
cleaner.set_strategy(FillMode())
df_mode = cleaner.clean(df_loaded)
print(f"\n--- Strategy 3: FillMode --- (Shape: {df_mode.shape})")
print("Remaining NaNs across all columns:", df_mode.isna().sum().to_dict())
display(df_mode)

Original Data Missing Values:
{'id': 0, 'name': 0, 'age': 3, 'height_cm': 1, 'weight_kg': 1, 'city': 0, 'score': 2}

--- Strategy 1: DropMissing --- (Shape: (2, 7))


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,5,Eva,27.0,160.0,55.0,New York,88.0



--- Strategy 2: FillMean --- (Shape: (7, 7))
Remaining numeric NaNs: {'id': 0, 'age': 0, 'height_cm': 0, 'weight_kg': 0, 'score': 0}


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0



--- Strategy 3: FillMode --- (Shape: (7, 7))
Remaining NaNs across all columns: {'id': 0, 'name': 0, 'age': 0, 'height_cm': 0, 'weight_kg': 0, 'city': 0, 'score': 0}


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


---
## Exercise 3: Add Decorators for Logging and Timing

### Objectives & Concepts
- **Decorator Pattern:** Extends the behavior of individual functions/methods dynamically without modifying their source code.
- **Cross-Cutting Concerns:** Logging and profiling are non-functional requirements that should not pollute business/cleaning logic.
- **Open/Closed Principle:** By using Python decorators (`@functools.wraps`), we wrap methods transparently.

### Instructions:
1. Create `@log_action`: logs when a method starts and finishes.
2. Create `@log_time`: calculates and prints execution time.
3. Apply them to methods in `CSVReader` or `DataCleaner`.

In [22]:
def _method_name(func, args):
    if args and hasattr(type(args[0]), func.__name__):
        return f"{type(args[0]).__name__}.{func.__name__}"
    return func.__name__


def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        qualname = _method_name(func, args)
        print(f"[LOG ACTION] --> Starting '{qualname}'...")
        result = func(*args, **kwargs)
        print(f"[LOG ACTION] <-- Finished '{qualname}' successfully.")
        return result
    return wrapper


def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        qualname = _method_name(func, args)
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f"[LOG TIME]   Execution time for '{qualname}': {elapsed:.6f} seconds")
        return result
    return wrapper

In [23]:
# Applying Decorators to CSVReader and DataCleaner methods

class DecoratedCSVReader(CSVReader):
    @log_time
    @log_action
    def read(self) -> pd.DataFrame:
        return super().read()

    @log_action
    def preview(self, n: int = 5) -> pd.DataFrame:
        return super().preview(n)


class DecoratedDataCleaner(DataCleaner):
    @log_time
    @log_action
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return super().clean(df)

In [24]:
# Testing Decorators in action
print("=== Testing DecoratedCSVReader ===")
dec_reader = DecoratedCSVReader("sample_data.csv")
data_dec = dec_reader.read()
dec_reader.preview(3)

print("\n=== Testing DecoratedDataCleaner ===")
dec_cleaner = DecoratedDataCleaner(FillMean())
cleaned_dec = dec_cleaner.clean(data_dec)

=== Testing DecoratedCSVReader ===
[LOG ACTION] --> Starting 'DecoratedCSVReader.read'...
[LOG ACTION] <-- Finished 'DecoratedCSVReader.read' successfully.
[LOG TIME]   Execution time for 'DecoratedCSVReader.read': 0.001254 seconds
[LOG ACTION] --> Starting 'DecoratedCSVReader.preview'...
=== Preview (First 3 rows of 'sample_data.csv') ===


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN


[LOG ACTION] <-- Finished 'DecoratedCSVReader.preview' successfully.

=== Testing DecoratedDataCleaner ===
[LOG ACTION] --> Starting 'DecoratedDataCleaner.clean'...
[LOG ACTION] <-- Finished 'DecoratedDataCleaner.clean' successfully.
[LOG TIME]   Execution time for 'DecoratedDataCleaner.clean': 0.001491 seconds


---
## Exercise 4: Implement Factory Pattern for Data Transformations

### Objectives & Concepts
- **Factory Pattern:** Encapsulates object creation logic. Clients specify *what* transformation they need (e.g. via a string type or configuration) rather than directly invoking concrete constructors.
- **DataTransform Hierarchy:** An abstract interface `DataTransform` with an `apply(df)` method.
- **Extensibility:** New transformations can be registered and instantiated seamlessly without modifying client code.

### Instructions:
1. Create an abstract class `DataTransform` with method `apply(df)`.
2. Implement subclasses:
   - `NormalizeColumns`: Scales numerical columns to $[0, 1]$ using Min-Max scaling:
     $$\tilde{x} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
   - `RemoveDuplicates`: Drops duplicate rows from the dataset.
   - `StandardizeText`: Cleans text/string columns (strips whitespace, title cases text).
3. Create a `TransformFactory` that returns transformation objects based on string input.

In [25]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies transformation logic to the given DataFrame.
        """
        pass


class NormalizeColumns(DataTransform):
    def __init__(self, columns: Optional[List[str]] = None):
        self.columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        transformed = df.copy()
        if self.columns is not None:
            target_cols = [c for c in self.columns if c in transformed.columns]
        else:
            target_cols = [
                c for c in transformed.select_dtypes(include=[np.number]).columns
                if c.lower() != 'id'
            ]

        for col in target_cols:
            col_min = transformed[col].min()
            col_max = transformed[col].max()
            if col_max > col_min:
                transformed[col] = (transformed[col] - col_min) / (col_max - col_min)
            else:
                transformed[col] = 0.0
        return transformed


class RemoveDuplicates(DataTransform):

    def __init__(self, subset: Optional[List[str]] = None):
        self.subset = subset

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates(subset=self.subset).reset_index(drop=True)


class StandardizeText(DataTransform):
    def __init__(self, columns: Optional[List[str]] = None, casing: str = 'title'):
        self.columns = columns
        self.casing = casing.lower()

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        transformed = df.copy()
        if self.columns is not None:
            target_cols = [c for c in self.columns if c in transformed.columns]
        else:
            target_cols = list(transformed.select_dtypes(include=['object', 'string']).columns)

        for col in target_cols:
            mask = transformed[col].notna()
            series = transformed.loc[mask, col].astype(str).str.strip()
            if self.casing == 'title':
                series = series.str.title()
            elif self.casing == 'lower':
                series = series.str.lower()
            elif self.casing == 'upper':
                series = series.str.upper()
            transformed.loc[mask, col] = series
        return transformed


class TransformFactory:
    _registry = {
        'normalize': NormalizeColumns,
        'normalize_columns': NormalizeColumns,
        'remove_duplicates': RemoveDuplicates,
        'standardize_text': StandardizeText,
    }

    @classmethod
    def register_transform(cls, name: str, transform_cls):
        """Registers a new transformation class dynamically."""
        cls._registry[name.lower()] = transform_cls

    @classmethod
    def get_transform(cls, transform_type: str, **kwargs) -> DataTransform:
        key = transform_type.strip().lower()
        transform_cls = cls._registry.get(key)
        if transform_cls is None:
            available = ", ".join(f"'{k}'" for k in cls._registry.keys())
            raise ValueError(f"Unknown transform '{transform_type}'. Available: [{available}]")
        return transform_cls(**kwargs)

In [26]:
# Testing TransformFactory and concrete transforms
print("=== Creating transforms via TransformFactory ===")
t_text = TransformFactory.get_transform('standardize_text', casing='title')
t_norm = TransformFactory.get_transform('normalize')
t_dedup = TransformFactory.get_transform('remove_duplicates')

print(f"Instantiated: {t_text.__class__.__name__}")
print(f"Instantiated: {t_norm.__class__.__name__}")
print(f"Instantiated: {t_dedup.__class__.__name__}")

# Applying transforms sequentially to the FillMean cleaned dataset
df_step1 = t_dedup.apply(df_mean)
df_step2 = t_text.apply(df_step1)
df_step3 = t_norm.apply(df_step2)

print("\nTransformed DataFrame Preview (Normalized numeric cols & Standardized text):")
display(df_step3)

=== Creating transforms via TransformFactory ===
Instantiated: StandardizeText
Instantiated: NormalizeColumns
Instantiated: RemoveDuplicates

Transformed DataFrame Preview (Normalized numeric cols & Standardized text):


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


---
## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objectives & Concepts
- **Template Method Pattern:** The base class defines the skeleton of the algorithm (`run()`), and subclasses implement the individual steps.
- **Composition:** The pipeline reuses `CSVReader`, `DataCleaner` and the `DataTransform` objects from the previous exercises.
- **Abstraction:** `@abstractmethod` forces every pipeline to implement `load`, `clean`, `transform` and `save`.

### Instructions:
1. Create an abstract class `DataPipeline` with a `run()` method that calls: `load()`, `clean()`, `transform()`, `save()`.
2. Create `CSVDataPipeline` that implements each step using the previous classes.

In [30]:
class DataPipeline(ABC):
    def run(self):                      # fixed order of steps
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        return df

    @abstractmethod
    def load(self): pass
    @abstractmethod
    def clean(self, df): pass
    @abstractmethod
    def transform(self, df): pass
    @abstractmethod
    def save(self, df): pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, cleaner, transforms):
        self.reader = CSVReader(input_path)
        self.output_path = output_path
        self.cleaner = cleaner
        self.transforms = transforms

    def load(self):
        return self.reader.read()

    def clean(self, df):
        return self.cleaner.clean(df)

    def transform(self, df):
        for t in self.transforms:
            df = t.apply(df)
        return df

    def save(self, df):
        df.to_csv(self.output_path, index=False)


pipeline = CSVDataPipeline(
    "sample_data.csv",
    "cleaned_sample_data.csv",
    DataCleaner(FillMean()),
    [TransformFactory.get_transform("remove_duplicates"),
     TransformFactory.get_transform("standardize_text"),
     TransformFactory.get_transform("normalize")],
)
result = pipeline.run()
print(result.round(2))

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  0.25       0.33       0.43     New York   0.59
1   2      Bob  0.41       0.80       0.50  Los Angeles   0.88
2   3  Charlie  1.00       0.53       0.57      Chicago   0.65
3   4    David  0.41       0.47       0.83      Houston   0.00
4   5      Eva  0.00       0.00       0.00     New York   0.76
5   6    Frank  0.38       1.00       1.00  Los Angeles   0.65
6   7    Grace  0.41       0.13       0.17      Chicago   1.00
